# Bayesian Performance Research

This is an exploratory research notebook for learning Bayesian statistics and developing the personal fitness-monitoring project step by step.

The notebook may contain templates, ideas, visual checks, simple baselines, and model experiments. It is not a production pipeline.

### Libraries & Data

Set up the research environment and data access here before starting a specific research question.

In [ ]:
# Libraries and helpers

import datetime
import importlib.util
import os
import sys
from pathlib import Path

import gspread
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm

# Optional Bayesian libraries
pymc_available = importlib.util.find_spec("pymc") is not None
arviz_available = importlib.util.find_spec("arviz") is not None
pm = None
az = None
if pymc_available:
    import pymc as pm
if arviz_available:
    import arviz as az

# Configs
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
# Set up project root path

current_path = Path.cwd().resolve()
project_root = next(
    (path for path in [current_path, *current_path.parents] if path.name == "bayesian_performance_job"),
    None,
)

if project_root is None:
    project_root = next(
        (path / "bayesian_performance_job" for path in [current_path, *current_path.parents]
         if (path / "bayesian_performance_job").is_dir()),
        None,
    )

if project_root is None:
    raise FileNotFoundError("Could not find the bayesian_performance_job directory.")

repository_root = project_root.parent
os.chdir(project_root)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Helpers
from daily_jobs import config
from daily_jobs import help_functions as hf
from daily_statistics_job import config as daily_config
from hasr_tl_job import config as hasr_config

# Environment check
print("Project root = {}".format(project_root))
print("Repository root for shared helpers = {}".format(repository_root))
print("PyMC available = {}".format(pymc_available))
print("ArviZ available = {}".format(arviz_available))

### Get Data

In [ ]:
# Get daily, activity, and HASR-TL data

user = "urh"
user_config = config.USER_CONFIGURATIONS[user]

googleDrive_client = gspread.authorize(config.DRIVE_CREDENTIALS)

daily_data, _ = hf.import_google_sheet(
    googleDrive_client = googleDrive_client,
    filename = user_config["gdrive_daily_log_filename"],
    sheet_name = config.BASIC_DAILY_STATISTICS_SHEET_NAME
    )
daily_data = hf.data_safe_convert_to_numeric(daily_data)

activity_data, _ = hf.import_google_sheet(
    googleDrive_client = googleDrive_client,
    filename = user_config["gdrive_activity_log_filename"],
    sheet_name = config.BASIC_ACTIVITY_STATISTICS_SHEET_NAME
    )
activity_data = hf.data_safe_convert_to_numeric(activity_data)

hasr_tl_data, _ = hf.import_google_sheet(
    googleDrive_client = googleDrive_client,
    filename = user_config["gdrive_activity_log_filename"],
    sheet_name = hasr_config.HASR_TL_SHEET_NAME
    )
hasr_tl_data = hf.data_safe_convert_to_numeric(hasr_tl_data)

### Prepare data

In [ ]:
# Prepare date columns

daily_data["Datetime"] = pd.to_datetime(
    daily_data["Year"].astype(str) + "-" +
    daily_data["Month"].astype(str) + "-" +
    daily_data["Day"].astype(str),
    format = "%Y-%m-%d",
    errors = "coerce"
    )

activity_data["Datetime"] = pd.to_datetime(
    activity_data["Year"].astype(str) + "-" +
    activity_data["Month"].astype(str) + "-" +
    activity_data["Day"].astype(str) + " " +
    activity_data["Start time"].fillna("00:00").astype(str),
    format = "%Y-%m-%d %H:%M",
    errors = "coerce"
    )

hasr_tl_data["Datetime"] = pd.to_datetime(
    hasr_tl_data["Year"].astype(str) + "-" +
    hasr_tl_data["Month"].astype(str) + "-" +
    hasr_tl_data["Day"].astype(str) + " " +
    hasr_tl_data["Start time"].fillna("00:00").astype(str),
    format = "%Y-%m-%d %H:%M",
    errors = "coerce"
    )

daily_data = daily_data.sort_values("Datetime").reset_index(drop = True)
activity_data = activity_data.sort_values("Datetime").reset_index(drop = True)
hasr_tl_data = hasr_tl_data.sort_values("Datetime").reset_index(drop = True)

### About the data

Inspect the available daily, activity, recovery, and training-load data before defining features or models.

### Research

Add one research question or statistical idea at a time. Record hypotheses, observations, interpretations, and decisions separately.

### Plots

Use focused plots to understand trends, variability, relationships, and possible measurement problems.

### Model diagnostic

Add model checks and uncertainty diagnostics once a model has been agreed and fitted.